<div style="
  background: linear-gradient(145deg, #0f172a, #1e293b);
  border: 4px solid transparent;
  border-radius: 14px;
  padding: 18px 22px;
  margin: 12px 0;
  font-size: 36px;
  font-weight: 600;
  color: #f8fafc;
  box-shadow: 0 6px 14px rgba(0,0,0,0.25);
  background-clip: padding-box;
  position: relative;
">
  <div style="
    position: absolute;
    inset: 0;
    padding: 4px;
    border-radius: 14px;
    background: linear-gradient(90deg, #06b6d4, #3b82f6, #8b5cf6);
    -webkit-mask: 
      linear-gradient(#fff 0 0) content-box, 
      linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor;
    mask-composite: exclude;
    pointer-events: none;
  "></div>
  <b>02. Bayesian Model Comparison</b>
</div>

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">📑 Table of Contents</span>

- **1. [Why Compare Models the Bayesian Way?](#-part-i-why-compare-models-the-bayesian-way)**
- **2. [Setup & Imports](#-part-ii-setup--imports)**
- **3. [Marginal Likelihood & Bayes Factors](#-part-iii-marginal-likelihood--bayes-factors)**
    - 3.1 [Computing Marginal Likelihood via MCMC](#computing-marginal-likelihood)
    - 3.2 [Bayes Factor Interpretation](#bayes-factor-interpretation)
- **4. [Information Criteria for Model Selection](#-part-iv-information-criteria)**
    - 4.1 [WAIC: Widely Applicable Information Criterion](#waic)
    - 4.2 [LOO-CV: Leave-One-Out Cross-Validation](#loo-cv)
- **5. [Practical Example: Polynomial Regression Selection](#-part-v-practical-example)**
- **6. [Key Takeaways](#-part-vi-key-takeaways)**

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">1. Why Compare Models the Bayesian Way?</span>

- Traditional model selection (AIC, BIC, cross-validation) treats model selection as a **separate step** from inference
- Bayesian model comparison provides a **principled, unified framework** based on probability theory

**Key ideas:**
- **Marginal likelihood** $p(\mathcal{D} | \mathcal{M})$ — how well does the model explain the data, integrating over all parameters?
- **Bayes factor** — ratio of marginal likelihoods between two models
- **Occam's razor is built in** — more complex models are automatically penalized

| **Method** | **Bayesian?** | **Handles Uncertainty?** | **Automatic Complexity Penalty?** |
|-----------|:---:|:---:|:---:|
| AIC | ❌ | ❌ | ⚠️ Approximate |
| BIC | ❌ | ❌ | ⚠️ Approximate |
| Cross-Validation | ❌ | ❌ | ✅ Empirical |
| **Bayes Factor** | **✅** | **✅** | **✅ Exact** |
| **WAIC** | **✅** | **✅** | **✅** |

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">2. Setup & Imports</span>

In [ ]:
import tensorflow as tf
import tensorflow_probability as tfp
import numpy as np
import matplotlib.pyplot as plt

tfd = tfp.distributions
tfb = tfp.bijectors

tf.random.set_seed(42)
np.random.seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print(f"TensorFlow: {tf.__version__}")
print(f"TFP: {tfp.__version__}")

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">3. Marginal Likelihood & Bayes Factors</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">3.1. Computing Marginal Likelihood</span>

The **marginal likelihood** (evidence) is:

$$p(\mathcal{D} | \mathcal{M}) = \int p(\mathcal{D} | \theta, \mathcal{M}) \, p(\theta | \mathcal{M}) \, d\theta$$

This integral averages the likelihood over all possible parameter values, weighted by the prior.

In [ ]:
# Generate synthetic data: y = 2x + 1 + noise
n_obs = 30
x_obs = np.linspace(-2, 2, n_obs).astype(np.float32)
y_obs = (2.0 * x_obs + 1.0 + np.random.normal(0, 0.5, n_obs)).astype(np.float32)

def compute_log_marginal_likelihood_mc(log_likelihood_fn, prior, num_samples=10000):
    """
    Monte Carlo estimate of the log marginal likelihood.
    log p(D) ≈ log(1/S * Σ p(D|θ_s)) where θ_s ~ p(θ)
    """
    # Sample from prior
    theta_samples = prior.sample(num_samples)
    
    # Compute log-likelihood for each sample
    log_likelihoods = tf.vectorized_map(log_likelihood_fn, theta_samples)
    
    # Log-mean-exp for numerical stability
    log_marginal = tf.reduce_logsumexp(log_likelihoods) - tf.math.log(tf.cast(num_samples, tf.float32))
    return log_marginal

# Model 1: Linear (y = ax + b)
prior_linear = tfd.Independent(
    tfd.Normal(loc=[0., 0.], scale=[5., 5.]), 1
)

def log_lik_linear(params):
    a, b = params[0], params[1]
    pred = a * x_obs + b
    return tf.reduce_sum(tfd.Normal(pred, 0.5).log_prob(y_obs))

# Model 2: Quadratic (y = ax^2 + bx + c)
prior_quad = tfd.Independent(
    tfd.Normal(loc=[0., 0., 0.], scale=[5., 5., 5.]), 1
)

def log_lik_quad(params):
    a, b, c = params[0], params[1], params[2]
    pred = a * x_obs**2 + b * x_obs + c
    return tf.reduce_sum(tfd.Normal(pred, 0.5).log_prob(y_obs))

lml_linear = compute_log_marginal_likelihood_mc(log_lik_linear, prior_linear)
lml_quad = compute_log_marginal_likelihood_mc(log_lik_quad, prior_quad)

print(f"Log marginal likelihood (Linear):    {lml_linear:.2f}")
print(f"Log marginal likelihood (Quadratic): {lml_quad:.2f}")

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">3.2. Bayes Factor Interpretation</span>

In [ ]:
# Bayes Factor: BF_12 = p(D|M1) / p(D|M2)
log_bf = lml_linear - lml_quad
bf = tf.exp(log_bf)

print(f"Log Bayes Factor (Linear vs Quadratic): {log_bf:.2f}")
print(f"Bayes Factor: {bf:.2f}")

# Interpretation table
print("\n" + "="*60)
print("Jeffreys' Scale for Interpreting Bayes Factors")
print("="*60)
interpretation = [
    ("BF > 100", "Decisive evidence for Model 1"),
    ("30 < BF < 100", "Very strong evidence for Model 1"),
    ("10 < BF < 30", "Strong evidence for Model 1"),
    ("3 < BF < 10", "Substantial evidence for Model 1"),
    ("1 < BF < 3", "Barely worth mentioning"),
    ("BF < 1", "Evidence favors Model 2"),
]
for range_str, interp in interpretation:
    print(f"  {range_str:20s} → {interp}")

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">4. Information Criteria for Model Selection</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">4.1. WAIC: Widely Applicable Information Criterion</span>

WAIC estimates the **out-of-sample predictive accuracy** using the posterior:

$$\text{WAIC} = -2 \left( \text{lppd} - p_{\text{WAIC}} \right)$$

Where:
- **lppd** = log pointwise predictive density (how well the model fits each data point)
- $p_{\text{WAIC}}$ = effective number of parameters (complexity penalty)

In [ ]:
def compute_waic(log_lik_matrix):
    """
    Compute WAIC from a log-likelihood matrix.
    
    Args:
        log_lik_matrix: shape [num_posterior_samples, num_data_points]
    
    Returns:
        waic, lppd, p_waic
    """
    # lppd: log pointwise predictive density
    S = tf.cast(tf.shape(log_lik_matrix)[0], tf.float32)
    lppd = tf.reduce_sum(
        tf.reduce_logsumexp(log_lik_matrix, axis=0) - tf.math.log(S)
    )
    
    # p_waic: effective number of parameters
    p_waic = tf.reduce_sum(tf.math.reduce_variance(log_lik_matrix, axis=0))
    
    # WAIC
    waic = -2.0 * (lppd - p_waic)
    
    return waic.numpy(), lppd.numpy(), p_waic.numpy()

# Get posterior samples via MCMC for both models
def get_posterior_samples(target_log_prob_fn, num_params, num_samples=2000):
    kernel = tfp.mcmc.SimpleStepSizeAdaptation(
        inner_kernel=tfp.mcmc.NoUTurnSampler(
            target_log_prob_fn=target_log_prob_fn,
            step_size=0.1
        ),
        num_adaptation_steps=400,
        target_accept_prob=0.75
    )
    
    @tf.function
    def run():
        return tfp.mcmc.sample_chain(
            num_results=num_samples,
            num_burnin_steps=500,
            current_state=tf.zeros(num_params),
            kernel=kernel,
            trace_fn=None
        )
    
    return run()

# Posterior for linear model
linear_post = get_posterior_samples(log_lik_linear, 2)

# Compute pointwise log-likelihood matrix
def pointwise_log_lik_linear(samples):
    log_liks = []
    for i in range(samples.shape[0]):
        pred = samples[i, 0] * x_obs + samples[i, 1]
        log_liks.append(tfd.Normal(pred, 0.5).log_prob(y_obs))
    return tf.stack(log_liks)

ll_matrix_linear = pointwise_log_lik_linear(linear_post)
waic_lin, lppd_lin, p_waic_lin = compute_waic(ll_matrix_linear)

print(f"Linear Model:")
print(f"  WAIC:   {waic_lin:.2f}")
print(f"  lppd:   {lppd_lin:.2f}")
print(f"  p_WAIC: {p_waic_lin:.2f}")
print(f"\n💡 Lower WAIC = better predictive performance")

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">5. Practical Example: Polynomial Regression Selection</span>

In [ ]:
# Compare polynomial models of degree 1 through 5
def make_design_matrix(x, degree):
    """Create polynomial design matrix."""
    return tf.stack([x**d for d in range(degree + 1)], axis=1)

def poly_log_posterior(params, X_design):
    """Log posterior for polynomial regression."""
    pred = tf.reduce_sum(X_design * params, axis=1)
    log_lik = tf.reduce_sum(tfd.Normal(pred, 0.5).log_prob(y_obs))
    log_prior = tf.reduce_sum(tfd.Normal(0., 5.).log_prob(params))
    return log_lik + log_prior

results = {}
for degree in range(1, 6):
    X_design = make_design_matrix(x_obs, degree)
    n_params = degree + 1
    
    # Get posterior samples
    target_fn = lambda p, Xd=X_design: poly_log_posterior(p, Xd)
    post_samples = get_posterior_samples(target_fn, n_params, num_samples=1500)
    
    # Compute pointwise log-likelihood
    ll_rows = []
    for i in range(post_samples.shape[0]):
        pred = tf.reduce_sum(X_design * post_samples[i], axis=1)
        ll_rows.append(tfd.Normal(pred, 0.5).log_prob(y_obs))
    ll_matrix = tf.stack(ll_rows)
    
    waic, lppd, p_waic = compute_waic(ll_matrix)
    results[degree] = {'waic': waic, 'lppd': lppd, 'p_waic': p_waic,
                       'samples': post_samples, 'X_design': X_design}
    
    print(f"Degree {degree}: WAIC={waic:.2f}, p_WAIC={p_waic:.2f}")

best_degree = min(results, key=lambda d: results[d]['waic'])
print(f"\n🏆 Best model: Degree {best_degree} (lowest WAIC)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# WAIC comparison
degrees = list(results.keys())
waics = [results[d]['waic'] for d in degrees]
colors = ['g' if d == best_degree else 'gray' for d in degrees]

axes[0].bar(degrees, waics, color=colors, edgecolor='white', linewidth=2)
axes[0].set_xlabel('Polynomial Degree', fontsize=14)
axes[0].set_ylabel('WAIC (lower is better)', fontsize=14)
axes[0].set_title('Model Comparison via WAIC', fontsize=16, fontweight='bold')
axes[0].set_xticks(degrees)

# Posterior predictive for best model
x_plot = np.linspace(-2.5, 2.5, 200).astype(np.float32)
X_plot = make_design_matrix(x_plot, best_degree)
best_samples = results[best_degree]['samples']

preds = []
for i in range(min(200, best_samples.shape[0])):
    pred = tf.reduce_sum(X_plot * best_samples[i], axis=1).numpy()
    preds.append(pred)
    axes[1].plot(x_plot, pred, color='m', alpha=0.03, linewidth=1)

preds = np.array(preds)
mean_pred = preds.mean(axis=0)

axes[1].plot(x_plot, mean_pred, color='b', linewidth=2.5, label='Posterior mean')
axes[1].plot(x_plot, 2.0 * x_plot + 1.0, 'r--', linewidth=2, label='True function')
axes[1].scatter(x_obs, y_obs, c='k', s=40, zorder=5, edgecolors='white', label='Data')
axes[1].set_xlabel('x', fontsize=14)
axes[1].set_ylabel('y', fontsize=14)
axes[1].set_title(f'Best Model (Degree {best_degree}) with Posterior Uncertainty',
                  fontsize=16, fontweight='bold')
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">6. Key Takeaways</span>

| **Method** | **When to Use** | **Pros** | **Cons** |
|-----------|----------------|----------|----------|
| Bayes Factor | Comparing 2–3 specific models | Exact, principled | Sensitive to priors |
| WAIC | Comparing many models | Easy to compute from MCMC | Asymptotic approximation |
| LOO-CV | Predictive performance focus | Robust, widely applicable | Computationally expensive |

### 🎯 Practical Guidelines
1. **Use WAIC** for routine model comparison — it's easy to compute from posterior samples
2. **Bayes Factors** are best when you have strong theoretical reasons for specific model comparisons
3. **Always check sensitivity** to prior choices, especially for Bayes Factors
4. **Lower WAIC/LOO** = better out-of-sample prediction
5. **Bayesian model averaging** can combine predictions from multiple models